In [155]:
import torch
import torchvision
import torch.nn as nn
from torchvision import transforms
from PIL import Image


In [156]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = torchvision.models.resnet18(pretrained=False)

num_feats = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_feats, 2)
)

model.load_state_dict(torch.load(r"D:\NCKH\last_model.pth", map_location=device))
model = model.to(device)
model.eval()


C:\Users\luan0\AppData\Local\Temp\ipykernel_17684\2147648859.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(r"D:\NCKH\last_model.pth",

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [157]:
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.Pad(64, fill=0),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])


In [158]:
def predict_image(img_path, model, transform, class_names):
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)  # (1, 3, 224, 224)

    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        pred_idx = probs.argmax(dim=1).item()

    return class_names[pred_idx], probs[0][pred_idx].item()


In [159]:
class_names = ['close', 'open']


test từng file ảnh


In [ ]:
# img_path = r"D:\NCKH\test\Screenshot 2026-02-09 221437.png"

# label, confidence = predict_image(
#     img_path,
#     model,
#     test_transform,
#     class_names
# )

# print(f"dự đoán: {label}")
# print(f"tin cậy: {confidence*100:.2f}%")

full folder

In [160]:

import os

test_dir = r"D:\NCKH\test"

for img_name in os.listdir(test_dir):
    if img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
        img_path = os.path.join(test_dir, img_name)
        label, conf = predict_image(
            img_path,
            model,
            test_transform,
            class_names
        )
        print(f"{img_name} → {label} ({conf*100:.1f}%)")


Screenshot 2026-02-09 221330.png → open (69.6%)
Screenshot 2026-02-09 221437.png → open (75.9%)
Screenshot 2026-02-09 221558.png → open (52.8%)


In [168]:
import os
import torch
import torchvision
import torch.nn as nn
from torchvision import transforms
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = r"D:\NCKH\ket_qua_option2\models\best_model.pth"
IMAGE_DIR = r"D:\NCKH\test"

class_to_idx = {'close': 0, 'open': 1}
idx_to_class = {v: k for k, v in class_to_idx.items()}

# ===== Load model =====
model = torchvision.models.resnet18(pretrained=False)
num_feats = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_feats, 2)
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# ===== Transform =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ===== Predict folder =====
for file in os.listdir(IMAGE_DIR):
    if file.lower().endswith(('.jpg', '.png', '.jpeg')):
        img_path = os.path.join(IMAGE_DIR, file)
        image = Image.open(img_path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)
            conf, pred = torch.max(probs, dim=1)

        print(f"{file:20s} -> {idx_to_class[pred.item()]} ({conf.item()*100:.2f}%)")


C:\Users\luan0\AppData\Local\Temp\ipykernel_17684\2817478848.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=D

Screenshot 2026-02-09 221330.png -> close (76.39%)
Screenshot 2026-02-09 221437.png -> open (96.14%)
Screenshot 2026-02-09 221558.png -> close (99.44%)
